# Section 2 - Retrieval Engine

Section 1 ended with the RAG idea: before asking the LLM, **retrieve** the few modules that matter and paste only those into the prompt. This section builds that retriever in three steps:

1. **Embedding:** turn text into numbers that capture meaning.
2. **Chunking:** decide which text gets turned into numbers.
3. **Retrieval:** find the modules closest to a question.

Cells marked **YOUR TURN** are for you to edit. Everything else runs as it is.

In [ ]:
# Standalone setup, for working on Section 2 on its own.
# Delete this cell when merging into workshop.ipynb: Section 1 already does all of it.
# %pip install -q sentence-transformers numpy matplotlib
import sys

sys.path.append("..")  # rag_utils.py is one folder up from notebooks/

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
print(len(modules), "modules, embeddings", embeddings.shape)

In [ ]:
# Two small helpers used throughout this section.
row_of = {module["moduleCode"]: i for i, module in enumerate(modules)}


def show(results):
    """Print (module, score) pairs, one per line."""
    for module, score in results:
        print(f"  {score:.2f}  {module['moduleCode']:<9} {module['title']}")

## 2.1 Embedding

In Section 1, Ctrl+F failed because it compares characters, and meaning is not in the characters. An **embedding model** reads a piece of text and returns a list of numbers, called a **vector**, that describes what the text is about. Texts with similar meanings get similar numbers.

We use `all-MiniLM-L6-v2`:

- **Small:** about 90 MB, and it runs on a laptop CPU.
- **Local:** no API key and no cost, because nothing is sent anywhere.
- **384 dimensions:** every text becomes 384 numbers, whether it is one word or a paragraph.

In [ ]:
vector = model.encode("Machine learning is fun")

print("Shape:", vector.shape)
print("First 8 numbers:", vector[:8].round(3))

The individual numbers mean nothing on their own. What matters is how close two vectors are.

We measure closeness with a **dot product**: multiply the two vectors position by position, then add everything up. With `normalize_embeddings=True` every vector has length 1, so the result is the cosine similarity. About 1 means the same meaning, and about 0 means unrelated.

In [ ]:
sentences = [
    "How do I learn to code?",
    "Where can I pick up programming?",
    "What should I eat for lunch?",
]
vectors = model.encode(sentences, normalize_embeddings=True)

print("Sentence 1 vs 2:", round(float(vectors[0] @ vectors[1]), 2))
print("Sentence 1 vs 3:", round(float(vectors[0] @ vectors[2]), 2))

Sentences 1 and 2 have no important word in common, yet they score far higher than 1 and 3. Ctrl+F could never tell you that.

### Real modules

Every module already has a vector. They were computed in advance and stored in `embeddings`, where row `i` belongs to `modules[i]`. Here are 10 modules from four areas, each compared with all the others.

In [ ]:
def plot_similarity(labels, vectors, title):
    """Heatmap of the similarity between every pair of vectors."""
    similarity = vectors @ vectors.T
    count = len(labels)

    fig, ax = plt.subplots(figsize=(9, 6))
    image = ax.imshow(similarity, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(count), range(1, count + 1))
    ax.set_yticks(range(count), [f"{label}  {i + 1}" for i, label in enumerate(labels)])
    for i in range(count):
        for j in range(count):
            colour = "white" if similarity[i, j] > 0.6 else "black"
            ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=8, color=colour)
    fig.colorbar(image, label="Similarity")
    ax.set_title(title)
    plt.show()


codes = [
    "CS2109S", "CS3244", "CS4248",     # AI
    "EC1101E", "FIN2704", "FIN3702A",  # economics and finance
    "LSM1301", "LSM2105",              # biology
    "EN1101E", "EN2210",               # literature
]
rows = [row_of[code] for code in codes]
labels = [f"{code} {modules[row]['title']}" for code, row in zip(codes, rows)]

plot_similarity(labels, embeddings[rows], "How similar are these modules?")

**How to read it:** each cell compares two modules, and a darker cell means more similar. The diagonal is every module against itself, so it is always 1.00.

The darker squares along the diagonal are the four areas: AI, economics and finance, biology, and literature. Nobody told the model which faculty a module belongs to. It grouped them from the text alone.

Unrelated modules do not score 0. They mostly sit around 0.1 to 0.3, so read the scores as a ranking: related pairs score higher than unrelated ones.

### Ctrl+F, again

Back to the query that returned nothing in Section 1.

In [ ]:
query = "how computers understand language"

print("Keyword search:", len(keyword_search(query)), "matches")
print("Semantic search:")
show(search(query))

Keyword search finds nothing, because no description contains that exact phrase. Semantic search returns modules about language, and the top hit is a course on how humans and machines handle language.

`search` is a black box for now. You will write it yourself in 2.3.

**Think about it:** most of the results are linguistics modules, not computing ones. Which words in the query pulled them in?

**YOUR TURN:** edit the sentences below and re-run the cell. Some things to try:

- The same word with two meanings: "I sat by the river bank" and "I opened a bank account".
- Opposites: "I love durian" and "I hate durian".
- The same sentence in two languages.

In [ ]:
# YOUR TURN: edit these sentences, then re-run
my_sentences = [
    "I love machine learning",
    "Neural networks are fascinating",
    "The stock market crashed today",
    "Investors are worried about inflation",
    "My cat sleeps all day",
    "Dogs are loyal pets",
]

plot_similarity(my_sentences, model.encode(my_sentences, normalize_embeddings=True), "My sentences")

## 2.2 Chunking

A **chunk** is one piece of text that gets its own vector. Real sources such as a handbook, a PDF or a website are too long to embed in one go, so we split them. There are two reasons:

- **Token budget:** MiniLM reads at most 256 tokens. Anything after that is silently cut off.
- **Precision:** one vector has to summarise the whole chunk. A vector for a 50-page handbook is about everything, so it is not close to any particular question.

NUSMods is a lucky case, because the data is already split for us: **one module = one chunk**. The decision left is which fields of a module go into the chunk text.

In [ ]:
module = modules[row_of["CS2040S"]]

print("The raw module:")
for field, value in module.items():
    print(f"  {field}: {value}")

print()
print("The chunk that gets embedded:")
print(build_chunk(module))

`build_chunk` (in `rag_utils.py`) joins four fields with line breaks: code, title, description and workload.

Look at the workload. NUSMods stores it as `[3, 0, 1, 3, 3]`, which tells an embedding model nothing, so we write it out in words first. Preparing text so that the model can use it is part of chunking.

How close are we to the token budget?

In [ ]:
chunks = [build_chunk(module) for module in modules]
lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print("CS2040S chunk:", lengths[row_of["CS2040S"]], "tokens. The limit is", model.max_seq_length)
print("Typical chunk:", int(np.median(lengths)), "tokens")

too_long = sum(length > model.max_seq_length for length in lengths)
print(f"{too_long} of {len(modules)} chunks are over the limit and lose their ending")

Only a small share of chunks get cut. When it happens, the end of the description and the workload are lost. That is why the code and title go first.

### What we left out

NUSMods has 21,160 modules on record for AY 2026-2027. We kept 6,552:

- **14,382 dropped** because they are not offered this year.
- **226 dropped** because the description is a placeholder such as "Not Available". With nothing to describe the module, its vector would come from the code and title alone.

### Which fields go in?

Adding a field is not free. To see why, we embed the same modules twice: once with the default chunk, and once with the prerequisites added.

Re-embedding all 6,552 modules takes minutes on a CPU, so we use the Computing modules only.

In [ ]:
computing = [module for module in modules if module["faculty"] == "Computing"]

fields_default = ["moduleCode", "title", "description", "workload"]
fields_prereq = fields_default + ["prerequisite"]

vectors_default = model.encode([build_chunk(m, fields_default) for m in computing], normalize_embeddings=True)
vectors_prereq = model.encode([build_chunk(m, fields_prereq) for m in computing], normalize_embeddings=True)

print(len(computing), "Computing modules embedded twice")
print()
print("A chunk with prerequisites:")
print(build_chunk(module, fields_prereq))

In [ ]:
def compare_chunks(query, needs=None):
    """Search the Computing modules with both versions of the chunk.

    `needs` is a module code. Results that list it as a prerequisite get a tick.
    """
    for name, vectors in [("Default chunk", vectors_default), ("With prerequisites", vectors_prereq)]:
        print(name)
        for result, score in search(query, embeddings=vectors, modules=computing):
            tick = "✓" if needs and needs in result.get("prerequisite", "") else " "
            print(f"  {score:.2f} {tick} {result['moduleCode']:<9} {result['title']}")


compare_chunks("I have completed CS2040S, what next?", needs="CS2040S")

A tick marks a module that really does list CS2040S as a prerequisite. With the prerequisites in the chunk, more of the top 5 are correct, because the text the question is about is now there to be matched.

Now a question about a topic.

In [ ]:
compare_chunks("I want to learn machine learning")

Every score dropped, and CS3244 Machine Learning, the module with exactly that title, fell out of the top 5. Its prerequisite text is long, and those words now share the vector with the topic.

That is the tradeoff: a field you add helps the questions about that field and dilutes everything else. Longer chunks also push more modules past the 256-token limit. Most questions are about topics, so the default chunk leaves prerequisites out.

**YOUR TURN:** choose your own fields below, re-embed the Computing modules, and compare with the default chunk. Does a shorter chunk do better or worse on your question?

In [ ]:
# YOUR TURN: choose the fields and the question, then re-run
# Fields you can use: moduleCode, title, description, workload, prerequisite, faculty, department
my_fields = ["title", "description"]
my_question = "I want to learn machine learning"

my_vectors = model.encode([build_chunk(m, my_fields) for m in computing], normalize_embeddings=True)

print("Default chunk")
show(search(my_question, embeddings=vectors_default, modules=computing))
print("My chunk:", my_fields)
show(search(my_question, embeddings=my_vectors, modules=computing))

## 2.3 Retrieval

Every module has a vector. To answer a question:

1. Embed the query with the same model.
2. Compare it with every module vector.
3. Sort by score.
4. Take the top `k`.

In [ ]:
def my_search(query, k=5):
    query_vector = model.encode(query, normalize_embeddings=True)  # 1. embed the query
    scores = embeddings @ query_vector                             # 2. compare with every module
    top = np.argsort(-scores)[:k]                                  # 3. sort, 4. take the top k
    return [(modules[i], float(scores[i])) for i in top]


show(my_search("how computers understand language"))

That is the whole retrieval engine, and it is what `search` in `rag_utils.py` does. The line `embeddings @ query_vector` is one matrix multiplication that computes all 6,552 dot products at once.

### Do we need a vector database?

In [ ]:
import time

query_vector = model.encode("how computers understand language", normalize_embeddings=True)

start = time.perf_counter()
scores = embeddings @ query_vector
top = np.argsort(-scores)[:5]
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Compared the query with {len(modules)} modules in {elapsed_ms:.1f} ms")

No. A vector database earns its place at millions of chunks, where comparing against everything becomes too slow. For a few thousand chunks, plain numpy is enough.

### Keyword search vs semantic search

Here are both, side by side, on the same query.

In [ ]:
def compare(query, k=5):
    """Show keyword search and semantic search for the same query."""
    matches = keyword_search(query)
    print(f"Keyword search: {len(matches)} matches")
    for match in matches[:k]:
        print(f"        {match['moduleCode']:<9} {match['title']}")
    print("Semantic search:")
    show(search(query, k))


compare("how computers understand language")

Semantic search wins that one. Now a query where the exact word matters.

In [ ]:
compare("ChatGPT")

Keyword search wins this one. Four descriptions mention ChatGPT by name, and Ctrl+F finds all of them. Semantic search finds one, then fills the other slots with communication modules.

Why? MiniLM was trained before ChatGPT existed, so it has never seen the word. This is how its tokenizer reads it:

In [ ]:
print(model.tokenizer.tokenize("ChatGPT"))

The model sees "chat" plus two fragments, so it goes looking for modules about communication. The embedding model has a knowledge cutoff too, just like the LLM in Section 1.

| | Keyword search | Semantic search |
|---|---|---|
| Matches | exact words | similar meaning |
| Good at | names, rare terms, new words | paraphrases and vague questions |
| Fails when | you use different words from the text | the exact word matters, or the model has never seen it |
| Returns | every match, in no particular order | always `k` results, ranked by score |

The last row matters: semantic search **always** returns `k` modules, even when nothing relevant exists.

**YOUR TURN:** ask your own questions. Some things to try:

- A topic in your own words, the way you would ask a senior.
- A module code, such as `CS2040S`.
- A greeting, such as `hi`.
- A filter, such as `4 MC, no exam, semester 2`.
- A larger `k`. Do the extra results still look relevant?

**Keep your funniest failure.** We will collect them in Section 3.

In [ ]:
# YOUR TURN: ask your own question, and change k
my_query = "I want to learn how to invest"
k = 5

compare(my_query, k)

## Recap

- **Embedding:** a model turns text into a vector, and similar meanings land close together.
- **Chunking:** one module is one chunk, and the fields you put in the chunk decide what it can be found by.
- **Retrieval:** embed the query, take a dot product with every module, sort, and keep the top `k`.

Section 3 takes the modules returned by `search(query, k=5)` and hands them to Gemini.